In [ ]:
import sys
from pathlib import Path
import importlib

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

from src.seed import SEED
# Top-to-bottom switches. A later stage reads files already on disk,
# so it can stay off when that input was saved earlier.
RUN = {
    "rebuild": False,
    "baseline": True,
    "train_embedding": True,
    "predict_embedding": True,
    "analytics": True,
    # IMPORTANT! Do not switch on unless necessary; it is extremely time-consuming.
    "load_stacking": False,
    "train_stacking": True,
    "predict_stacking": True,
}

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
# 16GB machine: a 6g driver plus every core writing a deep forest gets the JVM killed.
spark = SparkSession.builder \
    .master("local[4]") \
    .appName("DrugReviewClassification") \
    .config("spark.driver.memory", "8g") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
df_raw = data_oi.load_raw_data()
df_raw.show(truncate= 80)
print("rows:", df_raw.count())

### Exploratory Data Analysis (EDA)

Begin by trying to comprehend what your dataset conveys.

- What type of data are you currently exploring?
- What is the granularity of the data?
- What does the distribution of the data look like? Are there any outliers? Are there any missing or invalid entries?
- Is there any correlation between the variables you are interested in exploring?
- How would you cleanly and accurately visualize the relationship among variables?

In [ ]:
from src.eda import EDA
if RUN["rebuild"]:
    eda_raw = EDA(df_raw)
    eda_raw.dataframe_overview()

#### Fields

Each row is one comment on a drug, with the condition and a satisfaction score. The fields mix free text, categories, integers, and a date.

| Field | Type | Content |
|---|---|---|
| `uniqueID` | integer | Review identifier. There is no patient identifier. |
| `drugName` | string | Drug name as written. |
| `condition` | string | Condition or indication. |
| `review` | string | Comment text. Some values contain HTML entities such as `&#039;` and surrounding quotation marks. |
| `rating` | integer | Satisfaction score from 1 to 10. |
| `date` | date | Publication date. |
| `usefulCount` | integer | Helpful votes accumulated after publication. |


In [ ]:
if RUN["rebuild"]: eda_raw.granularity()

#### Granularity

The file has **161,297** reviews. `uniqueID` is unique, and the file has no patient identifier, so this is a review count.

- **3,436** distinct `drugName` strings. One comment can appear under more than one name, such as Sertraline and Zoloft.
- **885** distinct `condition` values, including null and text that is not a condition name.
- **48,635** groups share identical `review` text. **72** of those groups contain more than one `rating`.


In [ ]:
if RUN["rebuild"]: eda_raw.distribution_inspect()

#### Distributions and invalid values

`rating` stays inside 1–10. The scores pile up at the ends: **50,989** reviews (31.6%) are 10 and **21,619** (13.4%) are 1, while 2–7 are smaller. The standard deviation is 3.27 and the skewness is −0.80.

`usefulCount` has a long right tail (skewness 4.67). The median is 16, the third quartile is 36, and the maximum is 1,291. That maximum is one comment stored twice, under Sertraline and Zoloft.

Review length has a median of 455 characters and a maximum of 10,745. Six comments are three characters long. Some `condition` labels are truncated, including `Testicular Cance` and `min)`.

| Issue | Rows |
|---|---|
| Null `condition` | 899 (0.56%) |
| Helpful-vote text stored in `condition` | 900 (0.56%) |
| Nulls in the other six fields | 0 |
| Whitespace-only values | 0 |
| `rating` outside the integers 1–10 | 0 |
| Negative or non-numeric `usefulCount` | 0 |
| Dates that fail `d-MMM-yy`, or fall after 2026-11-02 | 0 |

The 900 `condition` values are page fragments such as `2</span> users found this comment helpful.`, separate from the 899 nulls. Comment text also contains HTML entities such as `&#039;`, and some condition names are cut off, such as `Panic Disorde` and `Stomach Ulce`.


In [ ]:
if RUN["rebuild"]: eda_raw.correlation()

#### Correlations

Pearson correlation uses `rating`, `usefulCount`, and `date` as days since 1970-01-01. A larger date value is a later review.

| Pair | r | Reading |
|---|---:|---|
| Rating, helpful votes | 0.234 | Higher ratings tend to receive somewhat more votes. |
| Rating, date | −0.196 | Later reviews have slightly lower ratings. |
| Helpful votes, date | −0.269 | Earlier reviews have more votes, consistent with a longer time to collect them. |

All three absolute values are below 0.3, so the linear associations are weak. `rating` is an ordinal score, and `usefulCount` has a long right tail. Drug and condition names stay categorical and are left out of this calculation.


In [ ]:
if RUN["rebuild"]: eda_raw.correlation_heatmap()

### Data Cleaning
A crucial step in the data preprocessing pipeline, aimed at improving the quality and accuracy of the dataset before analysis.
- Identifying Missing Values
- Handling Outliers
- Correcting Invalid Entries
- Removing Duplicates
- Standardizing Data: Ensure consistency in data representation.
- Transforming Variables: Apply necessary transformations to variables to enhance their interpretability or suitability for analysis.

In [ ]:
from src.cleaning import data_cleaning

if RUN["rebuild"]:
    print("Before data cleaning:")
    df_raw.show(10,truncate=80)
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    print("After data cleaning:")
    df_cleaned.show(10,truncate=80)
    print("Cleaned data overview:")
    eda_cleaned = EDA(df_cleaned)
    eda_cleaned.dataframe_overview()
    eda_cleaned.granularity()
    eda_cleaned.distribution_inspect()

In [ ]:
from src.text_features import text_feature_engineering

if RUN["rebuild"]:
    print("Before text feature engineering:")
    df_cleaned.show(10,truncate = 80)
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    print("After text feature engineering:")
    df_engineered.show(10,truncate = 80)

In [ ]:
from src.encoder import encoder
from src.split import split, weight


if RUN["rebuild"]:
    split_train, split_valid, split_test = split(df_engineered, SEED)
    weighted_train = weight(split_train)
    df_train, df_valid, df_test = encoder(weighted_train, split_valid, split_test)
    data_oi.save_split_data(df_train, df_valid, df_test, split_train)
    # weight() already released split_train. Keep the encoded frames for embedding.
    data_oi.unpersist(df_engineered, weighted_train, split_valid, split_test)
    if not RUN["baseline"]:
        data_oi.unpersist(df_cleaned)

df_train, df_valid, df_test = data_oi.load_split_data()

In [ ]:
from src.embeddings import EmbeddingModel
from src.text_features import baseline_text_feature_engineering
# Embedding Baseline Model (TFIDF without text feature engineering)

baseline_name = "tfidf"
improved_name = "tfidf_improved"

if RUN["baseline"]:
    if not RUN["rebuild"]:
        df_cleaned = data_cleaning(df_raw)

    with data_oi.scope():
        df_baseline_engineered = baseline_text_feature_engineering(df_cleaned)
        df_baseline_engineered.cache().count()
        data_oi.unpersist(df_cleaned)
        df_baseline_train, df_baseline_valid, df_baseline_test = split(df_baseline_engineered, SEED)
        df_baseline_train = weight(df_baseline_train)
        df_baseline_train, df_baseline_valid, df_baseline_test = encoder(
            df_baseline_train, df_baseline_valid, df_baseline_test
        )

        embedding_baseline_model = EmbeddingModel(df_baseline_train, df_baseline_valid, df_baseline_test)
        print(f"Embedding Baseline Model `{baseline_name}` Training")
        embedding_baseline_model.baseline_train(baseline_name)
        print(f"Embedding Baseline Model `{baseline_name}` Running")
        train_embedding_baseline, valid_embedding_baseline, test_embedding_baseline = embedding_baseline_model.baseline_predict(baseline_name)
        data_oi.save_embedding_data(
            baseline_name,
            train_embedding_baseline,
            valid_embedding_baseline,
            test_embedding_baseline,
        )

        df_train,df_valid,df_test = data_oi.load_split_data()
        improved_embedding_model = EmbeddingModel(df_train, df_valid, df_test)
        print(f"Embedding Baseline Model `{improved_name}` Training")
        improved_embedding_model.baseline_train(improved_name, {"vocabSize": 20000})
        print(f"Embedding Baseline Model `{improved_name}` Running")
        train_embedding_improved, valid_embedding_improved, test_embedding_improved = improved_embedding_model.baseline_predict(improved_name)
        data_oi.save_embedding_data(
            improved_name,
            train_embedding_improved,
            valid_embedding_improved,
            test_embedding_improved,
        )
        data_oi.unpersist(
            df_baseline_engineered,
            df_baseline_train, df_baseline_valid, df_baseline_test,
            train_embedding_baseline, valid_embedding_baseline, test_embedding_baseline,
            train_embedding_improved, valid_embedding_improved, test_embedding_improved,
        )
    del df_baseline_engineered, df_baseline_train, df_baseline_valid, df_baseline_test
    del embedding_baseline_model, improved_embedding_model
    del train_embedding_baseline, valid_embedding_baseline, test_embedding_baseline
    del train_embedding_improved, valid_embedding_improved, test_embedding_improved
    if not (RUN["train_embedding"] or RUN["predict_embedding"]):
        data_oi.unpersist(df_train, df_valid, df_test)
        del df_train, df_valid, df_test


In [ ]:
from src.embeddings import EmbeddingModel
from src.text_features import fit_phrases

# Phrases are fit on the training split. Prediction reuses this object.
embedding_model = EmbeddingModel(df_train, df_valid, df_test, fit_phrases(df_train))

if RUN["train_embedding"]:
    for name, train_function in EmbeddingModel.train_registry.items():
        print(f"Embedding Model `{name}` Training")
        train_function(embedding_model)

In [ ]:
if RUN["predict_embedding"]:
    for name, predict_function in EmbeddingModel.predict_registry.items():
        print(f"Embedding Model `{name}` Running")
        with data_oi.scope():
            train_embedding, valid_embedding, test_embedding = predict_function(embedding_model)
            data_oi.save_embedding_data(name, train_embedding, valid_embedding, test_embedding)

data_oi.unpersist(df_train, df_valid, df_test)
del embedding_model
del df_train, df_valid, df_test

In [ ]:
from src.analytics import AnalyticsModel

# Analytics Baseline Model (Logistic Regression Model without weight col)
if RUN["baseline"]:
    with data_oi.scope():
        train_embedding_baseline, valid_embedding_baseline, test_embedding_baseline = data_oi.load_embedding_data(baseline_name)
        analytics_baseline_model = AnalyticsModel(
            baseline_name, train_embedding_baseline, valid_embedding_baseline, test_embedding_baseline
        )
        print(f"Embedding `{baseline_name}` Analytics `lr`")
        train_pred, valid_pred, test_pred = analytics_baseline_model.baseline()
        data_oi.save_prediction_data(
            f"lr__{baseline_name}",
            train_pred,
            valid_pred,
            test_pred,
        )
        analytics_baseline_model.release_features()
        data_oi.unpersist(
            train_embedding_baseline, valid_embedding_baseline, test_embedding_baseline,
            train_pred, valid_pred, test_pred,
        )
        del train_pred, valid_pred, test_pred
        del train_embedding_baseline, valid_embedding_baseline, test_embedding_baseline, analytics_baseline_model

    with data_oi.scope():
        train_embedding_improved, valid_embedding_improved, test_embedding_improved = data_oi.load_embedding_data(improved_name)
        analytics_improved_model = AnalyticsModel(
            improved_name, train_embedding_improved, valid_embedding_improved, test_embedding_improved
        )
        print(f"Embedding `{improved_name}` Analytics lr_improved")
        train_pred, valid_pred, test_pred = analytics_improved_model.improved_baseline(
            data_oi.load_analytics_para("baseline", "improved_baseline")
        )
        data_oi.save_prediction_data(
            f"lr_improved__{improved_name}",
            train_pred,
            valid_pred,
            test_pred,
        )
        analytics_improved_model.release_features()
        data_oi.unpersist(
            train_embedding_improved, valid_embedding_improved, test_embedding_improved,
            train_pred, valid_pred, test_pred,
        )
        del train_pred, valid_pred, test_pred
        del train_embedding_improved, valid_embedding_improved, test_embedding_improved, analytics_improved_model


In [ ]:
# Analytics Model Prediction
from src.analytics import AnalyticsModel
from src.embeddings import EmbeddingModel

if RUN["analytics"]:
    for embedding_name in EmbeddingModel.predict_registry.keys():
        with data_oi.scope():
            train_embedding, valid_embedding, test_embedding = data_oi.load_embedding_data(embedding_name)
            analytics_model = AnalyticsModel(
                embedding_name,
                train_embedding,
                valid_embedding,
                test_embedding
            )
            for name, function in AnalyticsModel.registry.items():
                print(f"Embedding Model `{embedding_name}` Analytics Model `{name}`")
                # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.
                model_name = f"{name}__{embedding_name}"
                train_pred, valid_pred, test_pred = function(
                    analytics_model,
                    data_oi.load_analytics_para(embedding_name, name)
                )
                data_oi.save_prediction_data(model_name, train_pred, valid_pred, test_pred)
                analytics_model.release_features()
                del train_pred, valid_pred, test_pred


In [ ]:
from src.stacking import StackingModel

if RUN["load_stacking"]:
    StackingModel().load_data()

In [ ]:
from src.stacking import StackingModel

if RUN["train_stacking"]:
    StackingModel().train()

In [ ]:
from src.stacking import StackingModel

if RUN["predict_stacking"]:
    with data_oi.scope():
        predictions = StackingModel().predict()
        # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.
        data_oi.save_prediction_data("stacking__stacking", *predictions)
        del predictions

In [ ]:
from src.evaluation import Scorer

train_pred_dict, valid_pred_dict, test_pred_dict = data_oi.load_prediction_data()

scorer_train = Scorer(train_pred_dict)
scorer_valid = Scorer(valid_pred_dict)
scorer_test = Scorer(test_pred_dict)

print("Training Data:")
scorer_train.show_metrics(truncate = 40)

print("Validation Data:")
scorer_valid.show_metrics(truncate = 40)


In [ ]:
print("Testing Data:")
scorer_test.show_metrics(truncate = 50)